
# 🩻🐈‍⬛ KneeFusion-X3 V10 — Deadlock-Safe Champion Candidate

**RSNA Knee Abnormality Detection — stability-first challenger to V7 Public LB 0.782.**

### Why V9 stopped
V9 materialized a `float16` cache with shape `(4407, 6, 6, 252, 252)`, roughly **20.15 GB**, then used persistent DataLoader workers and reloaded DINOv2 for each fold. That combination is a plausible RAM / worker-lifecycle deadlock surface.

### V10 decisions
- preserve the strong **6 acquisition slots × 2 positions = 12 2.5D tokens/study**
- preserve **DINOv2-small**, target-conditioned anatomy routing, pathology-family experts and report-semantic auxiliary transfer
- **224 px** (native multiple of DINO patch size 14)
- cache MRI pixels as **uint8 disk memmap** instead of float16 RAM
- **no DataLoader, no multiprocessing workers, no persistent workers**
- load the offline DINOv2 checkpoint **once**, deep-copy it safely per fold
- 4/5 report-hash-safe folds with rank-mean test ensemble
- per-target official-label override; partial official labels are never discarded
- hard runtime guard + frequent heartbeats + Python stack watchdog
- always create a valid `submission.csv` before expensive work

**Required Kaggle input:** competition data + offline HuggingFace DINOv2-small (auto-detected under `/kaggle/input`; the Meta Kaggle model path used by V9 is supported).  
**Internet:** OFF. **GPU:** required.  
**Promotion rule:** this is a champion *candidate*, not a guaranteed leaderboard winner. Promote only if OOF is coherent and Public LB > 0.782.


In [ ]:

from __future__ import annotations
import os, re, gc, json, time, math, random, hashlib, warnings, unicodedata
import copy, shutil, faulthandler
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SEED = 20260808
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("V10 is a GPU notebook. Enable a Kaggle GPU accelerator.")
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEV = torch.device("cuda")
AMP = True
T0 = time.time()

TARGETS = [
    "ACL","MCL","Medial Meniscus","Lateral Meniscus",
    "Medial OA","Lateral OA","PF OA","Effusion",
    "Synovitis","Baker's","Contusion","Fracture"
]
ID = "StudyInstanceUID"

# Stability/performance controls.
IMG = 224
CROP_MM = 130.0
GROUP = 3
N_POS = 2
CACHE_SLICES = GROUP * N_POS
N_FOLDS = 5
FOLDS_TO_RUN = (0, 1, 2, 3)
EPOCHS = 4
BATCH = 2
ACCUM = 2

REPORT_DIM = 48
TFIDF_DIM = 24000
UNFREEZE_LAST = 4
BACKBONE_LR = 1.5e-5
HEAD_LR = 2.0e-4
WD = 1e-4
REPORT_LAMBDA = 0.10
GOLD_WEIGHT = 4.0
MIN_GOLD_AUC_TARGETS = 8
EARLY_STOPPING_PATIENCE = 2

# Pixel decode may use threads, but model batching uses ZERO subprocesses.
PIX_THREADS = min(8, os.cpu_count() or 4)
TRAIN_CUTOFF = 4.30 * 3600
HARD_CUTOFF = 4.65 * 3600
HEARTBEAT_STEPS = 200

def elapsed():
    return time.time() - T0

def log(msg):
    print(f"[{elapsed():8.1f}s] {msg}", flush=True)

# If any external library call blocks, Kaggle logs will still receive a traceback.
faulthandler.enable()
faulthandler.dump_traceback_later(15 * 60, repeat=True)

def find_root():
    for p in [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]:
        if (p / "train.csv").exists() and (p / "test.csv").exists():
            return p
    for p in Path("/kaggle/input").rglob("train.csv"):
        if (p.parent / "test.csv").exists() and (p.parent / "train_series.csv").exists():
            return p.parent
    raise FileNotFoundError("Competition root not found.")

ROOT = find_root()
print("ROOT:", ROOT)
print("DEVICE:", DEV, "| GPU:", torch.cuda.get_device_name(0))
print("V10 | img", IMG, "| folds", FOLDS_TO_RUN, "| epochs", EPOCHS, "| DataLoader workers = 0")


## 1. Report supervision
Official target values override weak labels **target-by-target**.

In [ ]:

# --- Multilingual report weak-label engine ---
_PRE = str.maketrans({
    "ı":"i","İ":"i","I":"i","ß":"ss","đ":"d","Đ":"d",
    "ø":"o","Ø":"o","æ":"ae","Æ":"ae"
})

def norm_text(x):
    if not isinstance(x, str):
        return ""
    x = x.translate(_PRE).lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[_/\\\-]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

NEG = re.compile(
    r"\b(no|not|without|none|absence|absent|negative for|unremarkable|intact|preserved|normal|"
    r"sin|no hay|ausencia|pas de|sans|aucun|geen|zonder|keine|ohne|yok|izlenmedi|saptanmadi|"
    r"nema|bez|δεν|χωρις|без|няма)\b"
)
UNC = re.compile(r"\b(possible|probable|suspect|suspicious|may|equivocal|muhtemel|supheli|moguc|πιθαν|verdacht|вероят)\w*")
TEAR = re.compile(r"\b(tear|torn|ruptur|rupture|sprain|injur|lesion|yirtik|desgarro|dechir|scheur|riss|pukn|ρηξ|разкъс|disruption|avuls)\w*")
OA = re.compile(r"\b(osteoarth|arthros|gonarth|chondrop|chondromal|cartilage loss|cartilage thinning|osteophyt|joint space narrowing|artrose|arthrose|kondrop|hondromal|αρθρωσ|остеофит|артроз)\w*")

ANAT = {
    "ACL": re.compile(r"\b(acl|anterior cruciate|cruzado anterior|croise anterieur|voorste kruisband|vorderes kreuzband|on capraz|prednji krizni|προσθι.*χιαστ|предна кръстна)\b"),
    "MCL": re.compile(r"\b(mcl|medial collateral|tibial collateral|colateral medial|collateral medial|mediales kollateral|ic yan bag|medijalni kolateral|εσω πλαγι|медиален колатерал)\b"),
    "Medial Meniscus": re.compile(r"\b(medial menisc|menisco medial|menisque medial|mediale meniscus|innenmeniskus|medyal menisk|medijalni menisk|εσω μηνισκ|медиал.*менискус)\b"),
    "Lateral Meniscus": re.compile(r"\b(lateral menisc|menisco lateral|menisque lateral|laterale meniscus|aussenmeniskus|lateral menisk|lateralni menisk|εξω μηνισκ|латерал.*менискус)\b"),
}
COMP = {
    "Medial OA": re.compile(r"\b(medial (femorotibial|tibiofemoral|compartment|condyle|plateau)|femorotibial medial|medial kompart|εσω διαμερισμα|медиал.*(компарт|отдел))\b"),
    "Lateral OA": re.compile(r"\b(lateral (femorotibial|tibiofemoral|compartment|condyle|plateau)|femorotibial lateral|lateral kompart|εξω διαμερισμα|латерал.*(компарт|отдел))\b"),
    "PF OA": re.compile(r"\b(patellofemoral|femoropatell|retropatell|trochle|trocle|patella|rotul|μηροεπιγονατιδ|феморопател)\w*"),
}
DIRECT = {
    "Effusion": re.compile(r"\b(effusion|joint fluid|hydrops|derrame|epanchement|gewrichtsvocht|gelenkerguss|efuzyon|sivi|izljev|ενδαρθρικ.*υγρ|излив)\w*"),
    "Synovitis": re.compile(r"\b(synovit|sinovit|synovial thick|synovial prolifer|pannus|υμενιτι|синовит)\w*"),
    "Baker's": re.compile(r"\b(baker|popliteal cyst|quiste poplite|kyste poplite|poplitealzyste|popliteal kist|bakerova|κυστη baker|бейк.*киста)\w*"),
    "Contusion": re.compile(r"\b(contusion|bone bruise|marrow edema|marrow oedema|bone marrow edema|edema oseo|botoedeem|knochenmarkodem|kemik.*odem|kostani edem|οστικο οιδημα|костномозъчен едем)\w*"),
    "Fracture": re.compile(r"\b(fractur|fractura|fractuur|fraktur|breuk|kirik|prijelom|καταγμα|фрактур|счупван|stress fracture|avulsion fracture)\w*"),
}

def clauses(report):
    t = norm_text(report)
    return [s.strip() for s in re.split(r"(?<=[.;!?])\s+|\n+", t) if s.strip()]

def score_clause_set(cls, anatomy, pathology=None):
    pos = neg = unc = 0
    for c in cls:
        if not anatomy.search(c):
            continue
        if pathology is not None and not pathology.search(c):
            continue
        if NEG.search(c):
            neg += 1
        elif UNC.search(c):
            unc += 1
        else:
            pos += 1
    if pos:
        return min(.95, .78 + .04 * min(pos, 3)), min(1., .70 + .08 * pos)
    if unc:
        return .58, .45
    if neg:
        return max(.04, .18 - .03 * min(neg, 3)), min(.9, .55 + .08 * neg)
    return .28, .05

def extract_report(report):
    cls = clauses(report)
    out = {}
    for t in TARGETS:
        if t in ANAT:
            s, c = score_clause_set(cls, ANAT[t], TEAR)
        elif t in COMP:
            s, c = score_clause_set(cls, COMP[t], OA)
        else:
            s, c = score_clause_set(cls, DIRECT[t], None)
        out[t] = s
        out[t + "__conf"] = c

    # Non-compartment-specific OA evidence: low-confidence fallback only.
    if any(OA.search(c) and not NEG.search(c) for c in cls):
        for t in ("Medial OA", "Lateral OA", "PF OA"):
            if out[t + "__conf"] < .10:
                out[t], out[t + "__conf"] = .62, .25
    return out

def build_weak_labels(train):
    # Prefer a previously validated mounted pseudo-label table if present.
    for p in Path("/kaggle/input").rglob("report_labels*.csv"):
        try:
            q = pd.read_csv(p)
            if ID in q.columns and all(t in q.columns for t in TARGETS):
                for t in TARGETS:
                    if t + "__conf" not in q.columns:
                        q[t + "__conf"] = 0.70
                print("LABEL SOURCE: mounted", p)
                q[ID] = q[ID].astype(str)
                return q.set_index(ID)
        except Exception:
            pass

    print("LABEL SOURCE: compact multilingual lexicon")
    q = pd.DataFrame([extract_report(x) for x in train["Report"].fillna("")])
    q[ID] = train[ID].astype(str).values
    return q.set_index(ID)


## 2. DICOM indexing and six acquisition slots

In [ ]:

# --- Competition tables and DICOM series metadata ---
train_df = pd.read_csv(ROOT / "train.csv")
test_df = pd.read_csv(ROOT / "test.csv")
train_df[ID] = train_df[ID].astype(str)
test_df[ID] = test_df[ID].astype(str)

# Fail-safe submission exists before any expensive work.
submission = test_df[[ID]].copy()
for t in TARGETS:
    submission[t] = 0.5
submission.to_csv("submission.csv", index=False)

trs = pd.read_csv(ROOT / "train_series.csv")
tes = pd.read_csv(ROOT / "test_series.csv")
for q in (trs, tes):
    q[ID] = q[ID].astype(str)
    q["SeriesInstanceUID"] = q["SeriesInstanceUID"].astype(str)

all_series = pd.concat([trs, tes], ignore_index=True)
plane_map = dict(zip(
    all_series["SeriesInstanceUID"],
    all_series["Anatomical_Plane"].astype(str).str.lower()
))

def series_dirs(split):
    out = []
    for dp, _, fn in os.walk(ROOT / split):
        fs = [str(Path(dp) / f) for f in fn if f.lower().endswith((".dcm", ".dicom"))]
        if fs:
            out.append((dp, fs))
    return out

HDR = [
    "StudyInstanceUID","SeriesInstanceUID","SeriesDescription","SequenceName",
    "ScanOptions","ScanningSequence","Laterality","PixelSpacing",
    "ImagePositionPatient","ImageOrientationPatient"
]

def probe(item):
    dp, files = item
    try:
        d = pydicom.dcmread(
            files[len(files)//2], stop_before_pixels=True,
            specific_tags=HDR, force=True
        )
        def g(k, default=""):
            v = getattr(d, k, default)
            if isinstance(v, (list, tuple, pydicom.multival.MultiValue)):
                return "|".join(map(str, v))
            return str(v)
        return {
            "dir": dp, "files": files,
            "study": g("StudyInstanceUID"),
            "series": g("SeriesInstanceUID"),
            "desc": " ".join([
                g("SeriesDescription"), g("SequenceName"),
                g("ScanOptions"), g("ScanningSequence")
            ]).lower(),
            "lat": g("Laterality").upper(),
            "spacing": g("PixelSpacing"),
            "ipp": g("ImagePositionPatient"),
            "iop": g("ImageOrientationPatient"),
            "n": len(files)
        }
    except Exception:
        return None

def annotate(split):
    items = series_dirs(split)
    log(f"{split}: {len(items)} series dirs")
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as ex:
        meta = [m for m in ex.map(probe, items) if m]
    return meta

def seq_group(desc):
    fs = bool(re.search(r"\b(fs|fat.?sat|fatsat|stir|spair)\b", desc))
    fluid = bool(re.search(r"\b(pd|proton|t2|stir)\b", desc))
    return "fluid" if (fs or fluid) else "struct"

SLOTS = [
    ("sag_fluid","sagittal","fluid"),
    ("sag_struct","sagittal","struct"),
    ("cor_fluid","coronal","fluid"),
    ("cor_struct","coronal","struct"),
    ("axi_fluid","axial","fluid"),
    ("axi_struct","axial","struct"),
]

def normalize_plane(x):
    x = str(x).lower()
    if "sag" in x: return "sagittal"
    if "cor" in x: return "coronal"
    if "axi" in x or "tra" in x: return "axial"
    return x

def pick_slots(meta):
    by = defaultdict(list)
    for m in meta:
        st, ser = m["study"], m["series"]
        pl = normalize_plane(plane_map.get(ser, ""))
        if pl not in ("sagittal", "coronal", "axial"):
            continue
        mm = dict(m)
        mm["plane"] = pl
        mm["group"] = seq_group(mm["desc"])
        by[st].append(mm)

    out = {}
    for st, ms in by.items():
        d = {}
        for name, pl, grp in SLOTS:
            cand = [m for m in ms if m["plane"] == pl and m["group"] == grp]
            if not cand and name.endswith("fluid"):
                cand = [m for m in ms if m["plane"] == pl]
            if cand:
                d[name] = max(
                    cand,
                    key=lambda m: m["n"] - 200 * bool(re.search(r"localizer|scout", m["desc"]))
                )
        out[st] = d
    return out

def parse_vec(s, n):
    try:
        a = np.array([float(x) for x in str(s).split("|")], dtype=np.float32)
        return a if len(a) >= n else None
    except Exception:
        return None

def side_from_meta(ms):
    tags = [m["lat"][:1] for m in ms if m.get("lat", "")[:1] in ("L", "R")]
    if tags:
        return Counter(tags).most_common(1)[0][0]
    xs = []
    for m in ms:
        p = parse_vec(m.get("ipp", ""), 3)
        if p is not None:
            xs.append(float(p[0]))
    return ("L" if np.median(xs) > 0 else "R") if xs else None

def order_files(files):
    rows = []
    for f in files:
        try:
            d = pydicom.dcmread(
                f, stop_before_pixels=True,
                specific_tags=["ImagePositionPatient","ImageOrientationPatient","InstanceNumber"],
                force=True
            )
            p = np.asarray(getattr(d, "ImagePositionPatient", []), dtype=float)
            o = np.asarray(getattr(d, "ImageOrientationPatient", []), dtype=float)
            if len(p) >= 3 and len(o) >= 6:
                z = float(np.dot(p[:3], np.cross(o[:3], o[3:6])))
            else:
                z = float(getattr(d, "InstanceNumber", len(rows)))
            rows.append((z, f))
        except Exception:
            rows.append((len(rows), f))
    return [f for _, f in sorted(rows)]

def read_slice(f, crop_mm=CROP_MM, out=IMG):
    d = pydicom.dcmread(f, force=True)
    a = d.pixel_array.astype(np.float32)
    a = a * float(getattr(d, "RescaleSlope", 1) or 1) + float(getattr(d, "RescaleIntercept", 0) or 0)
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        return np.zeros((out, out), np.float32)

    lo, hi = np.percentile(finite, [1, 99])
    hi = max(hi, lo + 1e-6)
    a = np.clip((a - lo) / (hi - lo), 0, 1)

    try:
        sp = getattr(d, "PixelSpacing", None)
        sy, sx = float(sp[0]), float(sp[1])
        hh = max(32, int(round(crop_mm / sy)))
        ww = max(32, int(round(crop_mm / sx)))
        cy, cx = np.array(a.shape) // 2
        y0 = max(0, cy - hh // 2)
        x0 = max(0, cx - ww // 2)
        a = a[y0:min(a.shape[0], y0+hh), x0:min(a.shape[1], x0+ww)]
    except Exception:
        pass

    return np.asarray(
        Image.fromarray((a * 65535).astype(np.uint16)).resize(
            (out, out), Image.Resampling.BILINEAR
        ),
        dtype=np.float32
    ) / 65535.0

def read_slot(files):
    fs = order_files(files)
    n = len(fs)
    if n < 1:
        return None
    centers = [int(round((n - 1) * q)) for q in (.38, .62)]
    ims = []
    for c in centers:
        ids = [max(0, min(n-1, c-1)), c, max(0, min(n-1, c+1))]
        ims.extend([read_slice(fs[j]) for j in ids])
    return np.stack(ims).astype(np.float32)

def laterality_normalize(x, plane, side):
    return x[..., ::-1].copy() if side == "R" and plane in ("coronal", "axial") else x

htr = annotate("train_series")
hte = annotate("test_series")
slots_tr = pick_slots(htr)
slots_te = pick_slots(hte)

meta_by_st = defaultdict(list)
for m in htr + hte:
    meta_by_st[m["study"]].append(m)

print("mean slots/train:", np.mean([len(slots_tr.get(s, {})) for s in train_df[ID]]))


## 3. Shape smoke test before expensive pixel decoding

In [ ]:

def gather_target_slot_bias(prior, slot_res, slot):
    # prior, slot_res: [K,S]; slot: [B,T] -> [B,K,T]
    if slot.ndim != 2:
        raise ValueError(f"slot must be [B,T], got {tuple(slot.shape)}")
    if slot.numel() and (int(slot.min()) < 0 or int(slot.max()) >= prior.shape[1]):
        raise IndexError("slot id outside available slot table")
    p = prior.T[slot].permute(0, 2, 1).contiguous()
    r = slot_res.T[slot].permute(0, 2, 1).contiguous()
    return p, r

_test_prior = torch.randn(len(TARGETS), len(SLOTS))
_test_res = torch.randn_like(_test_prior)
_test_slot = torch.tensor([
    [0,0,1,1,2,2,3,3,4,4,5,5],
    [5,5,4,4,3,3,2,2,1,1,0,0]
])
_p, _r = gather_target_slot_bias(_test_prior, _test_res, _test_slot)
assert _p.shape == (2, len(TARGETS), len(SLOTS) * N_POS)
assert _r.shape == _p.shape
print("✅ SHAPE SMOKE TEST:", tuple(_test_slot.shape), "->", tuple(_p.shape))


## 4. Decode once to **uint8 memmap**
V9's float16 RAM cache was ~20.15 GB. At 224 px, this cache is ~8 GB and lives on temporary disk.

In [ ]:

# --- Deadlock-safe uint8 memmap cache ---
CACHE_DIR = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/kaggle/working")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CACHE_PATH = CACHE_DIR / "kfx10_train_uint8.dat"
TEST_CACHE_PATH = CACHE_DIR / "kfx10_test_uint8.dat"

def planned_bytes(n):
    return int(n * len(SLOTS) * CACHE_SLICES * IMG * IMG)  # uint8

need = planned_bytes(len(train_df)) + planned_bytes(len(test_df))
free = shutil.disk_usage(CACHE_DIR).free
print("cache planned GB:", round(need / 1e9, 2), "| free GB:", round(free / 1e9, 2))
if free < need * 1.15:
    raise RuntimeError("Not enough temporary disk for deadlock-safe uint8 cache.")

def build_cache(frame, slot_map, tag, path):
    studies = frame[ID].astype(str).tolist()
    shape = (len(studies), len(SLOTS), CACHE_SLICES, IMG, IMG)
    cache = np.memmap(path, mode="w+", dtype=np.uint8, shape=shape)
    cache[:] = 0
    mask = np.zeros((len(studies), len(SLOTS)), dtype=np.float32)

    jobs = []
    for i, st in enumerate(studies):
        for k, (name, pl, _) in enumerate(SLOTS):
            m = slot_map.get(st, {}).get(name)
            if m:
                jobs.append((i, st, k, pl, m["files"]))

    log(f"{tag}: memmap {shape}, uint8; {len(jobs)} slot-series")

    def worker(j):
        i, st, k, pl, fs = j
        try:
            return i, st, k, pl, read_slot(fs)
        except Exception:
            return i, st, k, pl, None

    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as ex:
        for i, st, k, pl, x in ex.map(worker, jobs):
            done += 1
            if x is not None:
                x = laterality_normalize(x, pl, side_from_meta(meta_by_st.get(st, [])))
                cache[i, k] = np.rint(np.clip(x, 0, 1) * 255.0).astype(np.uint8)
                mask[i, k] = 1.0
            if done % 1024 == 0:
                cache.flush()
                log(f"{tag}: {done}/{len(jobs)}")
            if elapsed() > HARD_CUTOFF:
                raise RuntimeError("Hard runtime cutoff reached during DICOM cache build.")

    cache.flush()
    log(f"{tag}: filled {int(mask.sum())}/{len(jobs)}")
    return studies, cache, mask

st_tr, Ctr, Mtr = build_cache(train_df, slots_tr, "train", TRAIN_CACHE_PATH)
st_te, Cte, Mte = build_cache(test_df, slots_te, "test", TEST_CACHE_PATH)
print("cache ready:", Ctr.shape, Cte.shape, "| dtype:", Ctr.dtype, "| elapsed min:", round(elapsed()/60, 1))


## 5. Partial-gold supervision and report-hash-safe folds

In [ ]:

# --- Supervision and leakage-safe folds ---
lab = build_weak_labels(train_df)
raw = train_df.set_index(ID)[TARGETS].apply(pd.to_numeric, errors="coerce")
R = train_df.set_index(ID)["Report"].fillna("").to_dict()

Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)
W = np.zeros_like(Y)
G = np.full_like(Y, np.nan)

for i, uid in enumerate(st_tr):
    # weak report labels
    if uid in lab.index:
        r = lab.loc[uid]
        Y[i] = r[TARGETS].to_numpy(np.float32)
        conf = r[[t + "__conf" for t in TARGETS]].to_numpy(np.float32)
        W[i] = 0.20 + 0.80 * np.clip(conf, 0, 1)

    # official values override target-by-target, including partially labelled studies
    if uid in raw.index:
        g = raw.loc[uid].to_numpy(np.float32)
        G[i] = g
        m = np.isfinite(g)
        Y[i, m] = g[m]
        W[i, m] = GOLD_WEIGHT

print("official finite labels/target:", {
    t: int(np.isfinite(G[:, j]).sum()) for j, t in enumerate(TARGETS)
})

groups = np.array([
    hashlib.sha1(norm_text(R.get(st, "")).encode()).hexdigest()[:16]
    if norm_text(R.get(st, "")) else "uid_" + st
    for st in st_tr
])

fold_id = np.full(len(st_tr), -1, int)
for f, (_, va) in enumerate(GroupKFold(N_FOLDS).split(np.arange(len(st_tr)), groups=groups)):
    fold_id[va] = f
print("fold sizes:", Counter(fold_id))


## 6. Direct batch construction — zero subprocesses

In [ ]:

# --- Direct memmap batching: NO DataLoader, NO workers ---
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV)[None, None, :, None, None]
STD  = torch.tensor([0.229, 0.224, 0.225], device=DEV)[None, None, :, None, None]
SLOT_IDS = np.repeat(np.arange(len(SLOTS), dtype=np.int64), N_POS)

def make_batch(C, M, idx, train=False):
    idx = np.asarray(idx, dtype=np.int64)
    # [B,S,6,H,W] uint8 -> [B,S,2,3,H,W] -> [B,12,3,H,W]
    a = np.asarray(C[idx], dtype=np.uint8)
    b = len(idx)
    a = a.reshape(b, len(SLOTS), N_POS, 3, IMG, IMG)
    a = a.reshape(b, len(SLOTS) * N_POS, 3, IMG, IMG)
    x = torch.from_numpy(a).to(DEV, non_blocking=False).float().div_(255.0)

    if train:
        if random.random() < .35:
            gamma = random.uniform(.90, 1.10)
            x = x.clamp(0, 1).pow(gamma)
        if random.random() < .25:
            x = (x * random.uniform(.95, 1.05) + random.uniform(-.015, .015)).clamp(0, 1)

    x = (x - MEAN) / STD
    m = np.repeat(M[idx, :, None], N_POS, axis=2).reshape(b, -1)
    m = torch.from_numpy(m.astype(np.float32)).to(DEV)
    s = torch.from_numpy(np.tile(SLOT_IDS[None, :], (b, 1))).to(DEV)
    return x, m, s


## 7. DINOv2 + target-specific anatomy routing
The checkpoint is read from disk **once**; fold models are safe deep copies of the CPU template.

In [ ]:

# --- Anatomy-routed DINOv2 model ---
def find_dinov2():
    preferred = Path("/kaggle/input/models/metaresearch/dinov2/pytorch/small/1")
    if (preferred / "config.json").exists():
        return preferred
    for p in Path("/kaggle/input").rglob("config.json"):
        try:
            cfg = json.loads(p.read_text())
            mt = str(cfg.get("model_type", "")).lower()
            hs = int(cfg.get("hidden_size", 0))
            has_weights = any((p.parent / q).exists() for q in ("model.safetensors", "pytorch_model.bin"))
            if "dinov2" in mt and hs == 384 and has_weights:
                return p.parent
        except Exception:
            pass
    raise FileNotFoundError("Attach offline HuggingFace DINOv2-small weights.")

PLANE_PRIOR = {
    "ACL":[1.3,.8,.8,.4,.2,.1],
    "MCL":[.4,.3,1.4,.8,.2,.1],
    "Medial Meniscus":[1.1,.8,1.0,.7,.1,.1],
    "Lateral Meniscus":[1.1,.8,1.0,.7,.1,.1],
    "Medial OA":[.2,.5,.8,1.4,.1,.2],
    "Lateral OA":[.2,.5,.8,1.4,.1,.2],
    "PF OA":[.2,.4,.2,.4,1.4,.9],
    "Effusion":[.5,.2,.5,.2,1.3,.5],
    "Synovitis":[.5,.2,.5,.2,1.2,.5],
    "Baker's":[.9,.3,1.0,.3,.5,.2],
    "Contusion":[1.0,.4,1.0,.4,1.0,.4],
    "Fracture":[.8,.7,.8,.7,.8,.7],
}
FAMILY = {
    "ACL":0,"MCL":0,"Medial Meniscus":0,"Lateral Meniscus":0,
    "Medial OA":1,"Lateral OA":1,"PF OA":1,
    "Effusion":2,"Synovitis":2,"Baker's":2,
    "Contusion":3,"Fracture":3
}

class KneeFusionX3(nn.Module):
    def __init__(self, bb, dim, hidden=384, report_dim=REPORT_DIM):
        super().__init__()
        self.bb = bb
        self.hidden = hidden
        self.proj = nn.Sequential(
            nn.LayerNorm(dim * 3),
            nn.Linear(dim * 3, hidden),
            nn.GELU(),
            nn.Dropout(.10)
        )
        self.slot_emb = nn.Parameter(torch.randn(len(SLOTS), hidden) * .02)
        self.pos_emb = nn.Parameter(torch.randn(N_POS, hidden) * .02)
        self.query = nn.Parameter(torch.randn(len(TARGETS), hidden) * .02)
        self.cls_w = nn.Parameter(torch.randn(len(TARGETS), hidden) * .02)
        self.cls_b = nn.Parameter(torch.zeros(len(TARGETS)))
        self.slot_res = nn.Parameter(torch.zeros(len(TARGETS), len(SLOTS)))

        prior = torch.tensor([PLANE_PRIOR[t] for t in TARGETS], dtype=torch.float32)
        prior = (prior - prior.mean(1, keepdim=True)) * 0.8
        self.register_buffer("prior", prior)

        self.family_experts = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(hidden),
                nn.Linear(hidden, hidden),
                nn.GELU(),
                nn.Dropout(.10),
                nn.Linear(hidden, hidden)
            )
            for _ in range(4)
        ])
        self.family_gate = nn.Parameter(torch.zeros(len(TARGETS)))
        self.report_head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, report_dim)
        )

    def forward(self, x, mask, slot):
        B, T, C, H, W = x.shape
        z = self.bb(pixel_values=x.reshape(B*T, C, H, W)).last_hidden_state
        cls = z[:, 0]
        patch = z[:, 1:]
        pm = patch.mean(1)

        score = patch.norm(dim=-1)
        k = max(1, patch.shape[1] // 8)
        ix = score.topk(k, dim=1).indices
        focal = torch.gather(
            patch, 1, ix.unsqueeze(-1).expand(-1, -1, patch.shape[-1])
        ).mean(1)

        h = self.proj(torch.cat([cls, pm, focal], 1)).reshape(B, T, -1)
        pos = torch.arange(T, device=x.device) % N_POS
        h = h + self.slot_emb[slot] + self.pos_emb[pos].unsqueeze(0)

        att = torch.einsum("bth,kh->bkt", h, self.query) / (self.hidden ** .5)
        p, r = gather_target_slot_bias(self.prior, self.slot_res, slot)
        if p.shape != att.shape:
            raise RuntimeError(f"attention bias mismatch: att={att.shape}, prior={p.shape}")
        att = att + p + 0.35 * torch.tanh(r)
        att = att.masked_fill(mask.unsqueeze(1) < .5, -1e4)
        a = torch.softmax(att, -1)
        ctx = torch.einsum("bkt,bth->bkh", a, h)

        enriched = []
        for j, t in enumerate(TARGETS):
            rr = self.family_experts[FAMILY[t]](ctx[:, j])
            g = torch.sigmoid(self.family_gate[j])
            enriched.append(ctx[:, j] + g * rr)
        ctx = torch.stack(enriched, 1)

        logits = (ctx * self.cls_w.unsqueeze(0)).sum(-1) + self.cls_b
        den = mask.sum(1, keepdim=True).clamp_min(1)
        glob = (h * mask.unsqueeze(-1)).sum(1) / den
        return logits, self.report_head(glob), a

# Load DINO only once. Every fold deep-copies this CPU template; no repeated checkpoint I/O.
from transformers import AutoModel
DINO_PATH = find_dinov2()
log(f"DINOv2 path: {DINO_PATH}")
BASE_BB = AutoModel.from_pretrained(str(DINO_PATH), local_files_only=True).cpu()
for p in BASE_BB.parameters():
    p.requires_grad = False

def build_model(fold):
    torch.manual_seed(SEED + fold)
    bb = copy.deepcopy(BASE_BB)
    layers = bb.encoder.layer
    for block in layers[max(0, len(layers) - UNFREEZE_LAST):]:
        for p in block.parameters():
            p.requires_grad = True
    for p in bb.layernorm.parameters():
        p.requires_grad = True
    m = KneeFusionX3(bb, int(bb.config.hidden_size)).to(DEV)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"fold {fold}: trainable M={trainable/1e6:.2f}")
    return m


## 8. Four-fold training with hard cleanup between folds

In [ ]:

# --- Metrics and four-fold training ---
def binary_macro_auc(y, p):
    per, counts = {}, {}
    for j, t in enumerate(TARGETS):
        m = np.isfinite(y[:, j])
        yy, pp = y[m, j], p[m, j]
        counts[t] = int(m.sum())
        per[t] = (
            np.nan if len(yy) < 3 or len(np.unique(yy)) < 2
            else float(roc_auc_score(yy, pp))
        )
    vals = np.array(list(per.values()), float)
    n = int(np.isfinite(vals).sum())
    return (float(np.nanmean(vals)) if n else np.nan), per, counts, n

def weighted_soft_bce(y, p, w):
    p = np.clip(p, 1e-6, 1-1e-6)
    loss = -(y*np.log(p) + (1-y)*np.log(1-p))
    den = np.maximum(w.sum(0), 1e-6)
    per = (loss*w).sum(0) / den
    good = w.sum(0) > .05
    return float(per[good].mean())

def rank01(x):
    return pd.DataFrame(x).rank(method="average", pct=True).to_numpy(np.float32)

@torch.no_grad()
def predict(model, C, M, idx):
    model.eval()
    idx = np.asarray(idx, dtype=np.int64)
    out = []
    for s0 in range(0, len(idx), BATCH):
        ii = idx[s0:s0+BATCH]
        x, m, sl = make_batch(C, M, ii, train=False)
        with torch.autocast("cuda", enabled=AMP):
            z, _, _ = model(x, m, sl)
        out.append(torch.sigmoid(z).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)

history = []
member_test = []
trained_folds = []
oof = np.full_like(Y, np.nan)

def train_fold(f):
    tr = np.where(fold_id != f)[0]
    va = np.where(fold_id == f)[0]

    # Fold-safe report semantic teacher.
    tr_text = [R.get(st_tr[i], "") for i in tr]
    va_text = [R.get(st_tr[i], "") for i in va]
    tf = TfidfVectorizer(
        max_features=TFIDF_DIM, ngram_range=(1,2),
        min_df=2, sublinear_tf=True
    )
    A = tf.fit_transform(tr_text)
    dim = min(REPORT_DIM, max(2, min(A.shape) - 1))
    svd = TruncatedSVD(dim, random_state=SEED + f)
    Ztr = svd.fit_transform(A).astype(np.float32)
    if dim < REPORT_DIM:
        Ztr = np.pad(Ztr, ((0,0),(0,REPORT_DIM-dim)))
    pos_map = {int(i): k for k, i in enumerate(tr)}

    model = build_model(f)
    bbp = [p for n, p in model.named_parameters() if n.startswith("bb.") and p.requires_grad]
    hp = [p for n, p in model.named_parameters() if not n.startswith("bb.") and p.requires_grad]
    opt = torch.optim.AdamW(
        [{"params": bbp, "lr": BACKBONE_LR}, {"params": hp, "lr": HEAD_LR}],
        weight_decay=WD
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    best_selector = -1e9
    best_state = None
    best_epoch = 0
    stale = 0

    for ep in range(1, EPOCHS + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        order = tr.copy()
        np.random.default_rng(SEED + f*100 + ep).shuffle(order)

        tot = cls_tot = sem_tot = 0.0
        nstep = 0

        for step, s0 in enumerate(range(0, len(order), BATCH)):
            ix = order[s0:s0+BATCH]
            x, m, sl = make_batch(Ctr, Mtr, ix, train=True)
            yy = torch.from_numpy(Y[ix]).to(DEV)
            ww = torch.from_numpy(W[ix]).to(DEV)
            zz = torch.from_numpy(
                np.stack([Ztr[pos_map[int(i)]] for i in ix])
            ).to(DEV)

            with torch.autocast("cuda", enabled=AMP):
                lg, rp, _ = model(x, m, sl)
                bce = F.binary_cross_entropy_with_logits(lg, yy, reduction="none")
                # Normalize per target first: closer alignment with macro AUC.
                den = ww.sum(0).clamp_min(1e-6)
                cls = ((bce * ww).sum(0) / den).mean()
                sem = F.smooth_l1_loss(rp, zz)
                loss = (cls + REPORT_LAMBDA * sem) / ACCUM

            scaler.scale(loss).backward()
            if ((step + 1) % ACCUM == 0) or (s0 + BATCH >= len(order)):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

            tot += float(loss.item()) * ACCUM
            cls_tot += float(cls.item())
            sem_tot += float(sem.item())
            nstep += 1

            if step and step % HEARTBEAT_STEPS == 0:
                log(f"fold {f} ep {ep}: step {step}/{math.ceil(len(order)/BATCH)}")
            if elapsed() > TRAIN_CUTOFF:
                log("training cutoff reached inside epoch")
                break

        sched.step()

        pv = predict(model, Ctr, Mtr, va)
        val_soft_bce = weighted_soft_bce(Y[va], pv, W[va])
        gold_auc, gold_per, gold_n, n_auc = binary_macro_auc(G[va], pv)

        use_gold = np.isfinite(gold_auc) and n_auc >= MIN_GOLD_AUC_TARGETS
        selector = float(gold_auc) if use_gold else -float(val_soft_bce)
        selector_name = "gold_auc" if use_gold else "neg_soft_bce"

        row = {
            "fold": f, "epoch": ep,
            "train_loss": tot/max(nstep,1),
            "train_cls": cls_tot/max(nstep,1),
            "train_sem": sem_tot/max(nstep,1),
            "val_soft_bce": val_soft_bce,
            "gold_auc": gold_auc,
            "gold_auc_targets": n_auc,
            "selector": selector,
            "selector_name": selector_name,
            "elapsed_min": elapsed()/60
        }
        history.append(row)

        gold_txt = f"{gold_auc:.4f}" if np.isfinite(gold_auc) else "nan"
        print(
            f"fold {f} ep {ep}/{EPOCHS} | train {row['train_loss']:.4f} | "
            f"soft BCE {val_soft_bce:.4f} | gold AUC {gold_txt} ({n_auc}/{len(TARGETS)}) | "
            f"select={selector_name}:{selector:.4f} | {row['elapsed_min']:.1f} min",
            flush=True
        )

        if selector > best_selector + 1e-5:
            best_selector = selector
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = ep
            stale = 0
        else:
            stale += 1
            if stale >= EARLY_STOPPING_PATIENCE:
                print("early stop", flush=True)
                break

        if elapsed() > TRAIN_CUTOFF:
            break

    if best_state is None:
        raise RuntimeError(f"fold {f}: no checkpoint")

    model.load_state_dict(best_state)
    oof[va] = predict(model, Ctr, Mtr, va)
    tp = predict(model, Cte, Mte, np.arange(len(st_te)))

    torch.save({
        "fold": f,
        "state_dict": best_state,
        "targets": TARGETS,
        "version": "KneeFusion-X3-V10",
        "img": IMG,
        "crop_mm": CROP_MM,
        "slots": SLOTS,
        "n_pos": N_POS
    }, f"kfx3_v10_fold{f}.pth")

    # Explicitly release every fold-specific object before the next fold.
    del model, best_state, opt, sched, scaler, svd, tf, A, Ztr
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    log(f"fold {f} complete | best selector={best_selector:.5f} ep={best_epoch}")
    return tp

for f in FOLDS_TO_RUN:
    if elapsed() > TRAIN_CUTOFF:
        print("Runtime guard: stopping before next fold.", flush=True)
        break
    member_test.append(train_fold(f))
    trained_folds.append(f)

if not member_test:
    raise RuntimeError("No fold completed.")


## 9. OOF audit, rank ensemble and submission

In [ ]:

# --- Raw OOF audit + rank-mean submission ---
rows = np.where(np.isfinite(oof).any(1))[0]
if len(rows):
    soft_bce = weighted_soft_bce(Y[rows], oof[rows], W[rows])
    gold_auc, gold_per, gold_n, n_auc = binary_macro_auc(G[rows], oof[rows])
    print("OOF soft-label BCE:", round(soft_bce, 6))
    print(
        "OOF partial-gold macro AUC:",
        round(gold_auc, 6) if np.isfinite(gold_auc) else "nan",
        "| evaluable targets:", f"{n_auc}/{len(TARGETS)}"
    )
    pd.DataFrame({
        "target": TARGETS,
        "gold_auc": [gold_per.get(t, np.nan) for t in TARGETS],
        "gold_n": [gold_n.get(t, 0) for t in TARGETS]
    }).to_csv("oof_target_auc.csv", index=False)

pd.DataFrame(history).to_csv("history.csv", index=False)

# ROC AUC is rank-based; rank-mean reduces fold calibration drift.
ranks = np.stack([rank01(p) for p in member_test], 0)
ptest = ranks.mean(0)

pred = pd.DataFrame(ptest, columns=TARGETS)
pred[ID] = st_te
submission = test_df[[ID]].merge(pred, on=ID, how="left")
if submission[TARGETS].isna().any().any():
    miss = submission[TARGETS].isna().all(1).sum()
    raise RuntimeError(f"Missing predictions for {miss} rows.")

submission.to_csv("submission.csv", index=False)
print("trained folds:", trained_folds)
print("submission:", submission.shape, "| nulls:", int(submission[TARGETS].isna().sum().sum()))
print("total elapsed min:", round(elapsed()/60, 1))
display(submission.head())

# Remove multi-GB temporary caches so they are not persisted as Kaggle notebook output.
try:
    del Ctr, Cte
    gc.collect()
    for p in (TRAIN_CACHE_PATH, TEST_CACHE_PATH):
        if Path(p).exists():
            Path(p).unlink()
    print("temporary memmap caches removed")
except Exception as e:
    print("cache cleanup warning:", repr(e))

faulthandler.cancel_dump_traceback_later()



## Promotion gate

**Anchor:** V7 Public LB **0.782**.

Promote V10 only if:
1. all intended folds complete without worker/process hangs;
2. partial-gold OOF is not driven by one target family;
3. `history.csv` and `oof_target_auc.csv` show no target collapse;
4. Public LB exceeds **0.782** and agrees with the OOF direction.

### Expected engineering effect vs V9
- ~20.15 GB float16 RAM cache → ~8 GB uint8 disk memmap
- persistent DataLoader workers → **none**
- repeated DINO checkpoint reloads → **one**
- silent fold transition → explicit cleanup + CUDA synchronize + heartbeats
- 252 px → 224 px, reducing token compute while keeping DINO patch alignment

The ResNet-18 external checkpoint is **not needed** for this notebook.
